問題：撰寫一Python程式，為使用者輸入年份（可能是西元或民國）找尋其相對應的十二生肖。十二生肖是以十二年為一週期，每一年分別以一種動物來表示猴（monkey）、雞（rooster）、狗(dog)、豬(pig)、鼠(rat)、牛(ox)、虎(tiger)、兔(rabbit)、龍(dragon)、蛇（snake）、馬(horse)，以及羊(sheep) 。

請依序回答下面問題：

1. 0.1的二進位表示法為何? 使用markdown語法，列出計算過程和理由，並描述看到的現象。

0.1 在二進位中是一個無限循環小數:    
    計算            二進進位   說明
 1	0.1 × 2 = 0.2	  0	      0.2 < 1，位元為0
 2	0.2 × 2 = 0.4	  0	      0.4 < 1，位元為0
 3	0.4 × 2 = 0.8	  0	      0.8 < 1，位元為0
 4	0.8 × 2 = 1.6	  1	      1.6 ≥ 1，位元為1，取小數部分0.6
 5	0.6 × 2 = 1.2	  1	      1.2 ≥ 1，位元為1，取小數部分0.2
 6	0.2 × 2 = 0.4	  0	      回到步驟2，開始循環

 0.1₁₀ = 0.0001100110011001100110011...₂
 0.1₁₀ = 1.10011001100...₂ × 2⁻⁴
 看到的現象是 0.1 被存儲為略大於 0.1 的值（多了末尾的 555...）

2. 如果是無限多位，Python如何解決？提示：IEEE 754、四捨六入五成雙

IEEE 754 標準：
   Python 使用 64-bit (double precision) 浮點數
   結構：1位符號 + 11位指數 + 52位尾數 = 64位
   52位尾數提供約 15-17位十進位精度

「四捨六入五成雙」(Round Half to Even / Banker's Rounding)： 當尾數恰好是中間值（第53位是1，其後都是0）時：
   向最近的偶數舍入，不是總是向上或向下
   這減少了累積誤差

3. Python如何知道浮點數的精度？請問執行以下程式碼:
```python
import sys
print(sys.float_info)
```
並說明輸出結果的意思。

import sys
print(sys.float_info)

sys.float_info(max=1.7976931348623157e+308, max_exp=1024, max_10_exp=308, 
               min=2.2250738585072014e-308, min_exp=-1021, min_10_exp=-307, 
               dig=15, manidig=53, epsilon=2.220446049250313e-16, 
               radix=2, rounds=1)

屬性       值             意義
max	      1.798...e+308	 最大正浮點數
min	      2.225...e-308	 最小正浮點數
epsilon	  2.220...e-16	 機器精度：最小的 x，使得 1.0 + x ≠ 1.0
dig	      15	         十進位有效位數
manidig	  53	         尾數的二進位位數
radix	  2	             基數（二進位）
rounds	  1	             舍入方式（1=向最近偶數）

４. 在數學上0.1+0.2=0.3，可是下面的Python程式的答案好像不是0.3。怎麼解決？
```python
print(0.1+0.2)
```
問題原因： 0.1 和 0.2 都不能精確表示為二進位，累積誤差導致結果不等於 0.3

解決方案：
方案 A：使用 decimal 模組（推薦用於金融） 

from decimal import Decimal, getcontext

# 設置精度
getcontext().prec = 28

a = Decimal('0.1')
b = Decimal('0.2')
result = a + b
print(result)  # 輸出: 0.3
print(result == Decimal('0.3'))  # True

方案 B：使用 round() 函數（簡單場景）

result = round(0.1 + 0.2, 10)
print(result)  # 0.3
print(result == 0.3)  # True

方案 C：使用 fractions 模組（精確分數）

from fractions import Fraction

a = Fraction(1, 10)  # 1/10
b = Fraction(2, 10)  # 2/10
result = a + b
print(result)  # 3/10
print(float(result))  # 0.3

方案 D：相對誤差比較

import math

a, b, expected = 0.1, 0.2, 0.3
result = a + b
epsilon = 1e-15

is_equal = math.isclose(result, expected, rel_tol=epsilon)
print(is_equal)  # True

5. 你的方法，對於一般兩個浮點數加法，在科學或金融上是好方法嗎？請思考。

方案	            科學應用	金融應用	速度	評估
round()	            有限	   不適合	   快	   誤差仍然存在，不可靠
Decimal	            很好	   最佳	       較慢    金融必用，精度可控
Fraction            最佳	   很好	       很慢	   理論上完美，但計算開銷大
math.isclose()	    很好	   不適合	   快	   適合數值比較，不適合儲存
